# 22_performance: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/22_performance/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compare capacity, cost and replica consistency assumptions.
Run: uv run python modules/22_performance/examples/01_from_scratch.py
CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
Read the module's setup for optional extras and server requirements.
"""

from common.ops.capacity import estimate, monthly_cost
from common.ops.replication import ReplicaSimulation, shard_for


def main():
    for width in [4, 2, 1]:
        print(width, estimate(bytes_per_value=width, replicas=2))
    replicas = ReplicaSimulation()
    replicas.write("listing", "new price", acknowledgements=1)
    assert replicas.read(2, "listing") is None
    replicas.tick()
    assert replicas.read(2, "listing")[1] == "new price"
    replicas.alive = {0}
    try:
        replicas.write("listing", "later price", acknowledgements=2)
    except RuntimeError:
        print("Quorum unavailable: simulator, not a measured server outage")
    print("Shard routing:", [shard_for(str(i), 4) for i in range(12)])
    for hourly, hours in [(0.5, 8), (1.2, 2)]:
        parts = monthly_cost(
            hourly_compute=hourly,
            storage_gib=100,
            storage_rate=0.1,
            operator_hours=hours,
            operator_rate=75,
        )
        print("Hypothetical USD, not vendor prices:", parts, "total", sum(parts.values()))


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/22_performance/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Use a native clustering library to tune recall on development queries only.
Run: uv run python modules/22_performance/examples/02_production.py
CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
Read the module's setup for optional extras and server requirements.
"""

from common.ops.drills import target_tuning


def main():
    result = target_tuning()
    print(result)
    assert result["chosen_nprobe"] is not None
    assert result["heldout"]["samples"] == 30
    print("A held-out miss must be reported; never retune on the test split.")


if __name__ == "__main__":
    main()

## 03_measure

Source: `modules/22_performance/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure cold computation and tenant-aware cached retrieval.
Run: uv run python modules/22_performance/examples/03_measure.py
CPU: seconds to a minute; 2 GB RAM for frozen fixtures. No model downloads.
Read the module's setup for optional extras and server requirements.
"""

from common.ops.cache import QueryCache
from common.ops.drills import cache_measurement


def main():
    print(cache_measurement())
    cache = QueryCache()
    request = {"query": "chair", "filter": {"price": 100}, "k": 5}
    key = cache.key("a", "model-v1", 1, "search", request)
    cache.put(key, ["private-a"])
    variants = [
        cache.key("b", "model-v1", 1, "search", request),
        cache.key("a", "model-v2", 1, "search", request),
        cache.key("a", "model-v1", 2, "search", request),
        cache.key("a", "model-v1", 1, "search", {**request, "k": 10}),
    ]
    assert all(cache.get(k) is None for k in variants)
    print("Tenant, model, data generation and complete request change the key.")


if __name__ == "__main__":
    main()

## 04_cluster

Source: `modules/22_performance/examples/04_cluster.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None or "qdrant-cluster" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra qdrant; start qdrant-cluster and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Run the explicitly requested native qdrant-cluster experiment.
    Run: uv run --extra qdrant python modules/22_performance/examples/04_cluster.py
    CPU: about two minutes. Requires the Tier 5 runtime preparation and local setup.
    Owns and cleans only the processes or UUID pods created by this invocation.
    """
    # course-extra: qdrant
    # course-profile: qdrant-cluster

    from scripts.cluster_tier5 import main as run

    def main():
        run()

    if __name__ == "__main__":
        main()

## 05_load

Source: `modules/22_performance/examples/05_load.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("fastapi") is None or "api-load" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra apps; start api-load and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Run the explicitly requested native api-load experiment.
    Run: uv run --extra apps python modules/22_performance/examples/05_load.py
    CPU: about two minutes. Requires the Tier 5 runtime preparation and local setup.
    Owns and cleans only the processes or UUID pods created by this invocation.
    """
    # course-extra: apps
    # course-profile: api-load

    from scripts.load_tier5 import main as run

    def main():
        run()

    if __name__ == "__main__":
        main()

## 06_monitoring

Source: `modules/22_performance/examples/06_monitoring.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("cryptography") is None or "monitoring" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra ops; start monitoring and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Run the explicitly requested native monitoring experiment.
    Run: uv run --extra ops python modules/22_performance/examples/06_monitoring.py
    CPU: about two minutes. Requires the Tier 5 runtime preparation and local setup.
    Owns and cleans only the processes or UUID pods created by this invocation.
    """
    # course-extra: ops
    # course-profile: monitoring

    from scripts.monitor_tier5 import main as run

    def main():
        run()

    if __name__ == "__main__":
        main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.ops.capacity import estimate

print(estimate(rows=10000000, dimensions=768, replicas=2))
# Change precision, replicas and headroom separately.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.